# Random Forest



---

## What is Random Forest?

**Random Forest** is an ensemble algorithm that builds **many decision trees** and combines their outputs to make a final prediction.

It’s essentially:

> **Bagging + Decision Trees + Feature Randomness**

---

## Why “Random” Forest?

There are **two sources of randomness**, and both are crucial:

### 1. Random sampling of data (Bagging)

Each tree is trained on a **bootstrap sample** (sampling with replacement) of the training dataset.

➡️ This makes trees different from each other.

---

### 2. Random sampling of features

At **each split** in a tree:

* Only a **random subset of features** is considered
* Not all features compete for the best split

➡️ This decorrelates the trees.

Why this matters:
If one strong feature dominates, many trees would look the same. Feature randomness forces diversity.

---

## How Random Forest Works (Step by Step)

1. Draw `N` bootstrap samples from the dataset
2. Train one decision tree per sample
3. At each split:

   * Select `m` random features
   * Choose the best split among them
4. Aggregate predictions:

   * **Classification** → majority vote
   * **Regression** → average

---

## What Problem Does Random Forest Solve?

### Reduces variance

Decision trees:

* Powerful
* Highly unstable (small data change → very different tree)

Random Forest:

* Averages many trees
* Dramatically reduces overfitting

Bias stays roughly the same, variance drops → better generalization.

---

## Intuition

Think of each tree as a **noisy expert**.

One expert alone? Risky.
100 slightly wrong experts voting together? Surprisingly accurate.

---

## Key Hyperparameters (That Actually Matter)

### `n_estimators`

* Number of trees
* More trees → better performance (until it plateaus)
* Increases computation

### `max_depth`

* Controls tree complexity
* Smaller depth → less overfitting

### `max_features`

* Number of features considered per split
* Common defaults:

  * Classification: `sqrt(p)`
  * Regression: `p / 3`

### `min_samples_split` / `min_samples_leaf`

* Prevents tiny, overly specific splits
* Helps smooth predictions

---

## Out-of-Bag (OOB) Error (Very Cool Feature)

Because each tree sees only ~63% of data:

* The remaining ~37% can act as a **validation set**
* No need for cross-validation

This gives you:

* OOB accuracy
* Feature importance estimates

---

## Feature Importance

Random Forest estimates importance by:

* How much each feature reduces impurity across all trees

⚠️ Caveat:

* Biased toward high-cardinality features
* Permutation importance is often better

---

## Pros and Cons

### Pros

✅ Handles nonlinear relationships
✅ Works well out of the box
✅ Resistant to overfitting
✅ Handles mixed data types
✅ Minimal preprocessing

### Cons

❌ Less interpretable than a single tree
❌ Slower for large datasets
❌ Large memory footprint
❌ Not great at extrapolation (regression)

---

## Random Forest vs Decision Tree

| Aspect           | Decision Tree | Random Forest |
| ---------------- | ------------- | ------------- |
| Variance         | High          | Low           |
| Overfitting      | Common        | Rare          |
| Interpretability | High          | Medium-Low    |
| Accuracy         | Moderate      | High          |

---

## When Should You Use Random Forest?

* You want **strong performance quickly**
* Dataset is **tabular**
* Features interact in complex ways
* You don’t want to babysit hyperparameters

If gradient boosting is a sniper rifle, **Random Forest is a reliable shotgun**—less finicky, still deadly accurate.

---




Implementation:

In [ ]:
import numpy as np

class DecisionTreeRegressor:
    def __init__(self, max_depth=None, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.tree = None

    def mse(self, y):
        return np.mean((y - np.mean(y)) ** 2) if len(y) > 0 else 0

    def split(self, X, y, feature, threshold):
        mask = X[:, feature] <= threshold
        return X[mask], y[mask], X[~mask], y[~mask]

    def best_split(self, X, y):
        best_gain = 0
        best_feature = None
        best_threshold = None

        parent_mse = self.mse(y)
        n_features = X.shape[1]

        # Feature subsampling
        features = np.random.choice(
            n_features,
            self.max_features if self.max_features else n_features,
            replace=False
        )

        for feature in features:
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)

                if len(y_l) == 0 or len(y_r) == 0:
                    continue

                weighted_mse = (
                    len(y_l) / len(y) * self.mse(y_l)
                    + len(y_r) / len(y) * self.mse(y_r)
                )

                gain = parent_mse - weighted_mse

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def build_tree(self, X, y, depth=0):
        if (
            len(y) < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return np.mean(y)

        feature, threshold = self.best_split(X, y)
        if feature is None:
            return np.mean(y)

        X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(X_l, y_l, depth + 1),
            "right": self.build_tree(X_r, y_r, depth + 1),
        }

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    def _predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree
        if x[tree["feature"]] <= tree["threshold"]:
            return self._predict_one(x, tree["left"])
        return self._predict_one(x, tree["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])


In [ ]:
class RandomForestRegressor:
    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        max_features="sqrt"
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def _get_max_features(self, n_features):
        if self.max_features == "sqrt":
            return int(np.sqrt(n_features))
        if self.max_features == "log2":
            return int(np.log2(n_features))
        if isinstance(self.max_features, int):
            return self.max_features
        return n_features

    def fit(self, X, y):
        self.trees = []
        n_samples, n_features = X.shape
        max_feats = self._get_max_features(n_features)

        for _ in range(self.n_estimators):
            # Bootstrap sampling
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_feats
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(predictions, axis=0)


In [ ]:
import numpy as np
from collections import Counter

class DecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.tree = None

    # Gini impurity
    def gini(self, y):
        counts = Counter(y)
        impurity = 1.0
        total = len(y)
        for label in counts:
            p = counts[label] / total
            impurity -= p ** 2
        return impurity

    def split(self, X, y, feature, threshold):
        mask = X[:, feature] <= threshold
        return X[mask], y[mask], X[~mask], y[~mask]

    def best_split(self, X, y):
        best_gain = 0
        best_feature = None
        best_threshold = None
        parent_gini = self.gini(y)
        n_features = X.shape[1]

        # Feature subsampling
        features = np.random.choice(
            n_features,
            self.max_features if self.max_features else n_features,
            replace=False
        )

        for feature in features:
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)
                if len(y_l) == 0 or len(y_r) == 0:
                    continue
                weighted_gini = (len(y_l) / len(y) * self.gini(y_l)
                                + len(y_r) / len(y) * self.gini(y_r))
                gain = parent_gini - weighted_gini
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        return best_feature, best_threshold

    def build_tree(self, X, y, depth=0):
        if (len(set(y)) == 1 
            or len(y) < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)):
            return Counter(y).most_common(1)[0][0]

        feature, threshold = self.best_split(X, y)
        if feature is None:
            return Counter(y).most_common(1)[0][0]

        X_l, y_l, X_r, y_r = self.split(X, y, feature, threshold)
        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(X_l, y_l, depth+1),
            "right": self.build_tree(X_r, y_r, depth+1)
        }

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    def _predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree
        if x[tree["feature"]] <= tree["threshold"]:
            return self._predict_one(x, tree["left"])
        return self._predict_one(x, tree["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])


In [ ]:
class RandomForestClassifier:
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, max_features="sqrt"):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def _get_max_features(self, n_features):
        if self.max_features == "sqrt":
            return int(np.sqrt(n_features))
        if self.max_features == "log2":
            return int(np.log2(n_features))
        if isinstance(self.max_features, int):
            return self.max_features
        return n_features

    def fit(self, X, y):
        self.trees = []
        n_samples, n_features = X.shape
        max_feats = self._get_max_features(n_features)

        for _ in range(self.n_estimators):
            # Bootstrap sample
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_feats
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        # Collect predictions from all trees
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        # Majority vote
        final_preds = []
        for i in range(X.shape[0]):
            counts = Counter(all_preds[:, i])
            final_preds.append(counts.most_common(1)[0][0])
        return np.array(final_preds)


In [ ]:
import numpy as np
from collections import Counter

class RandomForestClassifier:
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, max_features="sqrt"):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []
        self.oob_samples = []  # store OOB indices
        self.feature_importances_ = None

    def _get_max_features(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if self.max_features == "log2":
            return max(1, int(np.log2(n_features)))
        if isinstance(self.max_features, int):
            return self.max_features
        return n_features

    def fit(self, X, y):
        n_samples, n_features = X.shape
        max_feats = self._get_max_features(n_features)

        self.trees = []
        self.oob_samples = []
        self.feature_importances_ = np.zeros(n_features)

        for _ in range(self.n_estimators):
            indices = np.random.choice(n_samples, n_samples, replace=True)
            oob_idx = np.setdiff1d(np.arange(n_samples), indices)
            self.oob_samples.append(oob_idx)

            X_sample = X[indices]
            y_sample = y[indices]

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_feats
            )
            tree.fit(X_sample, y_sample)

            self.trees.append(tree)

            # Compute feature importance for this tree
            self._accumulate_feature_importance(tree.tree, X_sample, y_sample)

        # Average feature importance
        self.feature_importances_ /= self.n_estimators

    def _accumulate_feature_importance(self, node, X, y):
        if not isinstance(node, dict):
            return

        # Current split info
        feature = node["feature"]
        threshold = node["threshold"]
        mask = X[:, feature] <= threshold
        y_left, y_right = y[mask], y[~mask]

        gini_parent = self._gini(y)
        gini_left = self._gini(y_left)
        gini_right = self._gini(y_right)
        weighted_gini = (len(y_left)/len(y)*gini_left + len(y_right)/len(y)*gini_right)
        importance = gini_parent - weighted_gini
        self.feature_importances_[feature] += importance

        # Recurse
        self._accumulate_feature_importance(node["left"], X[mask], y_left)
        self._accumulate_feature_importance(node["right"], X[~mask], y_right)

    def _gini(self, y):
        counts = Counter(y)
        impurity = 1.0
        total = len(y)
        if total == 0:
            return 0
        for label in counts:
            p = counts[label] / total
            impurity -= p ** 2
        return impurity

    def predict(self, X):
        # Aggregate predictions
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        final_preds = []
        for i in range(X.shape[0]):
            counts = Counter(all_preds[:, i])
            final_preds.append(counts.most_common(1)[0][0])
        return np.array(final_preds)

    def oob_score(self, X, y):
        n_samples = X.shape[0]
        votes = [[] for _ in range(n_samples)]

        for tree_idx, tree in enumerate(self.trees):
            oob_idx = self.oob_samples[tree_idx]
            if len(oob_idx) == 0:
                continue
            preds = tree.predict(X[oob_idx])
            for i, idx in enumerate(oob_idx):
                votes[idx].append(preds[i])

        oob_preds = np.zeros(n_samples, dtype=int)
        for i, v in enumerate(votes):
            if v:
                oob_preds[i] = Counter(v).most_common(1)[0][0]
            else:
                oob_preds[i] = -1  # no OOB prediction for this sample

        mask = oob_preds != -1
        return np.mean(oob_preds[mask] == y[mask])


**What OOB is:**

When a Random Forest builds each tree, it samples the training data with replacement (bootstrap sampling).

That means some samples are left out for that tree — these are the OOB samples for that tree.

On average, about 1/3 of the samples are OOB for any single tree


**How it works**

Train each tree on its bootstrap sample.

For each training sample, collect predictions from all trees where this sample was OOB.

Take a majority vote (classification) or average (regression) of these OOB predictions.

Compare to the true label → gives an unbiased estimate of generalization error.

Key point: you never use the sample in the tree’s training — so it’s like a mini hold-out set.